# GENSKETCH — training CNN su Sketchy

Notebook autonomo: non importa moduli da src. Riproduce la CNN a 4 blocchi e 2 convoluzioni per blocco, la divisione in 5 fold basata sulle foto, AdamW e la scelta del checkpoint tramite loss di validazione.

**Dati richiesti:** una directory con photo/<categoria>/*.jpg, sketch/<categoria>/*.png e, preferibilmente, index.json generato dal progetto. In locale usa ./data. Su Kaggle cerca questi file nella sorgente collegata sotto /kaggle/input/<nome-dataset>/, /data/ oppure una directory annidata come /GENSKETCH/. Se ci sono più sorgenti compatibili, imposta DATA_ROOT_OVERRIDE nella cella seguente.

I checkpoint, la configurazione, le metriche per epoca e il riepilogo sono salvati in runs/ in locale oppure in /kaggle/working/runs/ su Kaggle. Per eseguire un solo fold, modifica FOLDS_TO_RUN. Ogni fold parte da un modello e un ottimizzatore nuovi. Il test viene calcolato dopo aver caricato il checkpoint con la migliore loss di validazione.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import random
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

BASE_SEED = 42
K_FOLDS = 5
CAT_SUBSET = 10
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 225
PATIENCE = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
HIDDEN_LAYERS = 4
STARTING_HIDDEN_CHANNELS = 16
GROUPS = 4
FOLDS_TO_RUN = tuple(range(K_FOLDS))  # Per esempio: (0,)
DATA_ROOT_OVERRIDE = None             # Per esempio: Path("/kaggle/input/gensketch/GENSKETCH")

IS_KAGGLE = Path("/kaggle/input").is_dir()
OUTPUT_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
NUM_WORKERS = 2 if IS_KAGGLE else 0

def find_data_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        candidates = [Path(DATA_ROOT_OVERRIDE)]
    elif IS_KAGGLE:
        children = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
        candidates = []
        for child in children:
            candidates.extend((child, child / "data"))
            for nested in child.iterdir():
                if nested.is_dir():
                    candidates.extend((nested, nested / "data"))
    else:
        candidates = [Path.cwd() / "data"]

    matches = [p for p in candidates if (p / "photo").is_dir() and (p / "sketch").is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Attesa una directory con photo/ e sketch/, trovate: {matches}. "
            "Imposta DATA_ROOT_OVERRIDE."
        )
    return matches[0]

DATA_ROOT = find_data_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Device: {DEVICE}; PyTorch: {torch.__version__}")
if DEVICE.type != "cuda":
    print("GPU non disponibile: su Kaggle abilita l'acceleratore GPU nelle opzioni della sessione.")


## Indice e fold

Se index.json è presente, usa esattamente le etichette e i fold del progetto. Se manca, li ricostruisce dalle foto: il fold di ogni sketch coincide con quello della foto associata. Per mantenere gli ID globali delle 125 classi quando carichi solo alcune categorie, includi index.json originale.


In [ ]:
def build_index(root: Path, selected: list[str], k: int, seed: int) -> dict:
    categories = sorted(p.name for p in (root / "photo").iterdir() if p.is_dir())
    class_to_id = {name: i for i, name in enumerate(categories)}
    records = []

    for category in selected:
        label = class_to_id[category]
        photo_ids = [p.stem for p in sorted((root / "photo" / category).glob("*.jpg"))]
        if not photo_ids:
            raise ValueError(f"Nessuna foto JPG in {category}")
        random.Random(seed + label).shuffle(photo_ids)
        fold_by_photo = {photo_id: position % k for position, photo_id in enumerate(photo_ids)}

        for sketch in sorted((root / "sketch" / category).glob("*.png")):
            photo_id = sketch.stem.rsplit("-", 1)[0]
            if photo_id not in fold_by_photo:
                raise FileNotFoundError(f"Foto associata non trovata per {sketch}")
            records.append({
                "sketch": sketch.relative_to(root).as_posix(),
                "photo": (Path("photo") / category / f"{photo_id}.jpg").as_posix(),
                "photo_id": photo_id,
                "category": category,
                "label": label,
                "fold": fold_by_photo[photo_id],
            })
    return {"class_to_id": class_to_id, "records": records}

selected = sorted(p.name for p in (DATA_ROOT / "photo").iterdir() if p.is_dir())[:CAT_SUBSET]
if not selected:
    raise ValueError("Non ci sono categorie in photo/")

index_path = DATA_ROOT / "index.json"
if index_path.is_file():
    index = json.loads(index_path.read_text(encoding="utf-8"))
    print(f"Indice caricato: {index_path}")
else:
    index = build_index(DATA_ROOT, selected, K_FOLDS, BASE_SEED)
    print("index.json assente: indice ricostruito in memoria")

class_to_id = index["class_to_id"]
num_classes = max(class_to_id.values()) + 1
selected_set = set(selected)
records = [r for r in index["records"] if r["category"] in selected_set]
if not records or any(category not in class_to_id for category in selected):
    raise ValueError("Indice privo delle categorie selezionate")
if any(r["fold"] not in range(K_FOLDS) for r in records):
    raise ValueError("Indice con fold non validi")
if any(not (DATA_ROOT / r["sketch"]).is_file() for r in records):
    raise FileNotFoundError("Almeno uno sketch dell'indice manca nella directory dati")

print(f"Classi nell'output: {num_classes}; categorie allenate: {selected}")
print(f"Sketch selezionati: {len(records)}")
for category in selected:
    category_records = [r for r in records if r["category"] == category]
    print(f"{category}: {len(category_records)} sketch, "
          f"{len({r['photo_id'] for r in category_records})} foto")

def split_records(all_records: list[dict], test_fold: int) -> tuple[list, list, list]:
    val_fold = (test_fold + 1) % K_FOLDS
    train = [r for r in all_records if r["fold"] not in (test_fold, val_fold)]
    val = [r for r in all_records if r["fold"] == val_fold]
    test = [r for r in all_records if r["fold"] == test_fold]
    if not train or not val or not test:
        raise ValueError(f"Split vuoto nel fold {test_fold}")
    return train, val, test


## Dataset e rete

Il preprocessing riproduce quello locale: grayscale, resize a 256 × 256, tensore float in [0, 1]. Il numero di classi in uscita resta quello dell'indice globale.


In [ ]:
class SketchClassifierDataset(Dataset):
    def __init__(self, root: Path, rows: list[dict], size: int = IMAGE_SIZE):
        self.root = root
        self.rows = rows
        self.size = size

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int):
        row = self.rows[index]
        with Image.open(self.root / row["sketch"]) as image:
            array = np.array(
                image.convert("L").resize((self.size, self.size)),
                dtype=np.float32,
            )
        sketch = torch.from_numpy(array).unsqueeze(0) / 255.0
        return sketch, int(row["label"])

class CNNClassification(nn.Module):
    def __init__(self, in_channels: int, hidden_layers: int,
                 out_classes: int, starting_hidden_channels: int = 16):
        super().__init__()
        channels = starting_hidden_channels
        self.blocks = nn.ModuleList()
        for i in range(hidden_layers):
            self.blocks.append(nn.Sequential(
                nn.Conv2d(in_channels if i == 0 else channels // 2,
                          channels, kernel_size=3, stride=1, padding=1),
                nn.GroupNorm(num_groups=GROUPS, num_channels=channels),
                nn.LeakyReLU(),
                nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
                nn.GroupNorm(num_groups=GROUPS, num_channels=channels),
                nn.LeakyReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
                if i < hidden_layers - 1 else nn.AdaptiveAvgPool2d(output_size=1),
            ))
            channels *= 2
        self.flatten_layer = nn.Flatten()
        self.output_layer = nn.Linear(channels // 2, out_classes)

    def forward(self, sketches: torch.Tensor) -> torch.Tensor:
        out = sketches
        for block in self.blocks:
            out = block(out)
        return self.output_layer(self.flatten_layer(out))

model_check = CNNClassification(1, HIDDEN_LAYERS, num_classes, STARTING_HIDDEN_CHANNELS)
with torch.inference_mode():
    shape_check = model_check(torch.zeros(2, 1, IMAGE_SIZE, IMAGE_SIZE)).shape
assert shape_check == (2, num_classes), shape_check
print(f"Output di prova: {shape_check}; parametri: "
      f"{sum(p.numel() for p in model_check.parameters()):,}")
del model_check


## Training, validazione e checkpoint

La loss e l'accuracy sono medie pesate per numero di sketch. Ogni epoca è registrata sia in CSV sia in un file .log con il formato e gli eventi del training locale. Il test del fold viene eseguito una sola volta, dopo il caricamento del suo miglior checkpoint secondo la loss di validazione.


In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    loss_fn: nn.Module, optimizer: torch.optim.Optimizer) -> tuple[float, float]:
    model.train()
    total_loss = total_correct = total_samples = 0
    for sketches, labels in loader:
        sketches = sketches.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(sketches)
        if not torch.isfinite(logits).all():
            raise RuntimeError("Logits non finiti")
        loss = loss_fn(logits, labels)
        if not torch.isfinite(loss):
            raise RuntimeError("Loss non finita")
        loss.backward()
        optimizer.step()
        n = labels.size(0)
        total_loss += loss.item() * n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += n
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model: nn.Module, loader: DataLoader,
             loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    total_loss = total_correct = total_samples = 0
    with torch.inference_mode():
        for sketches, labels in loader:
            sketches = sketches.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(sketches)
            loss = loss_fn(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss non finita")
            n = labels.size(0)
            total_loss += loss.item() * n
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += n
    return total_loss / total_samples, total_correct / total_samples

config = {
    "base_seed": BASE_SEED, "k_folds": K_FOLDS, "categories": selected,
    "num_classes": num_classes, "image_size": IMAGE_SIZE,
    "hidden_layers": HIDDEN_LAYERS, "convs_per_block": 2,
    "starting_hidden_channels": STARTING_HIDDEN_CHANNELS,
    "groups": GROUPS, "activation": "LeakyReLU(0.01)",
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "patience": PATIENCE,
    "optimizer": "AdamW", "lr": LR, "weight_decay": WEIGHT_DECAY,
    "folds_to_run": FOLDS_TO_RUN,
}
config_hash = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
run_name = f"cnn_{config_hash}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = OUTPUT_ROOT / "runs" / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "config.json").write_text(
    json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Run directory: {RUN_DIR}")

# Stesso nome, hash di configurazione e formato di src/utils/logging/writing.py.
log_config = {
    "base_seed": BASE_SEED,
    "model": "CNN",
    "hidden_layers": HIDDEN_LAYERS,
    "convs_per_block": 2,
    "starting_hidden_channels": STARTING_HIDDEN_CHANNELS,
    "normalization": "GroupNorm",
    "groups": GROUPS,
    "categories": selected,
    "image_size": IMAGE_SIZE,
    "optimizer": "AdamW",
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "min_delta": None,
}
log_config_json = json.dumps(log_config, sort_keys=True, separators=(",", ":"))
log_config_id = hashlib.sha256(log_config_json.encode("utf-8")).hexdigest()[:10]
LOG_FILE = f"classification_main_{log_config_id}_{datetime.now(ZoneInfo('Europe/Rome')).strftime('%Y%m%d')}"
LOG_PATH = RUN_DIR / f"{LOG_FILE}.log"

def write_log(msg: str, module: str = LOG_FILE, lvl: str = "INFO") -> None:
    with LOG_PATH.open("a", encoding="utf-8") as stream:
        stream.write("[" + module + "]\t" + lvl + "\t| " + msg + "\n")

write_log(f"CONFIG_ID: {log_config_id}\nCONFIG: {log_config_json}")


In [ ]:
results = []
loss_fn = nn.CrossEntropyLoss()
start = datetime.now(ZoneInfo("Europe/Rome"))
write_log(f"RUN STARTED AT {start.strftime('%H:%M:%S')}")
csv_fields = ["fold", "epoch", "train_loss", "train_acc",
              "val_loss", "val_acc", "best_val_loss", "best_epoch"]

for test_fold in FOLDS_TO_RUN:
    if test_fold not in range(K_FOLDS):
        raise ValueError(f"Fold non valido: {test_fold}")
    fold_seed = BASE_SEED + test_fold
    torch.manual_seed(fold_seed)
    write_log(f"=====Fold {test_fold}=====\n=====[Seed] {fold_seed}=====\n")
    shuffle_generator = torch.Generator().manual_seed(fold_seed)
    train_rows, val_rows, test_rows = split_records(records, test_fold)

    def make_loader(rows: list[dict], shuffle: bool = False) -> DataLoader:
        return DataLoader(
            SketchClassifierDataset(DATA_ROOT, rows, IMAGE_SIZE),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            generator=shuffle_generator if shuffle else None,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        )

    train_loader = make_loader(train_rows, shuffle=True)
    val_loader = make_loader(val_rows)
    test_loader = make_loader(test_rows)
    model = CNNClassification(1, HIDDEN_LAYERS, num_classes,
                              STARTING_HIDDEN_CHANNELS).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_epoch = None
    stale_epochs = 0
    checkpoint_path = RUN_DIR / f"cnn_fold_{test_fold}_best.pt"
    history_path = RUN_DIR / f"fold_{test_fold}_history.csv"
    print(f"\nFold {test_fold} | seed {fold_seed} | "
          f"train {len(train_rows)}, val {len(val_rows)}, test {len(test_rows)}")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, stale_epochs = val_loss, epoch, 0
            write_log(f"NEW BEST VAL LOSS: {best_val_loss}")
            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "fold": test_fold,
                "config": config,
                "class_to_id": class_to_id,
            }, checkpoint_path)
        else:
            stale_epochs += 1

        row = {
            "fold": test_fold, "epoch": epoch,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "best_val_loss": best_val_loss, "best_epoch": best_epoch,
        }
        with history_path.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=csv_fields)
            if stream.tell() == 0:
                writer.writeheader()
            writer.writerow(row)
        print(f"Fold {test_fold} | {epoch:03d}/{EPOCHS} | "
              f"train {train_loss:.4f} / {train_acc:.3f} | "
              f"val {val_loss:.4f} / {val_acc:.3f} | "
              f"best {best_val_loss:.4f} @ {best_epoch}", flush=True)
        if stale_epochs >= PATIENCE:
            print(f"Early stopping al fold {test_fold}, epoca {epoch}")
            break
        write_log(
            f"[FOLD {test_fold}][EPOCH {epoch}]\n"
            f"TRAIN:\n"
            f"\tLOSS: {train_loss}\n"
            f"\tACC: {train_acc}\n"
            f"VALIDATION:\n"
            f"\tLOSS: {val_loss}\n"
            f"\t(BEST [{best_epoch}]): {best_val_loss}\n"
            f"\tACC: {val_acc}\n"
        )

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(checkpoint["model_state"])
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)
    result = {
        "fold": test_fold, "seed": fold_seed, "best_epoch": checkpoint["epoch"],
        "val_loss": checkpoint["val_loss"], "val_acc": checkpoint["val_acc"],
        "test_loss": test_loss, "test_acc": test_acc,
    }
    results.append(result)
    (RUN_DIR / "fold_results.json").write_text(
        json.dumps(results, indent=2), encoding="utf-8"
    )
    print(f"TEST fold {test_fold}: loss={test_loss:.4f}, accuracy={test_acc:.4f}", flush=True)
    write_log(
        f"FOLD: {test_fold}\n"
        f"TEST:\n"
        f"\tLOSS: {test_loss}\n"
        f"\tACC: {test_acc}\n"
    )

if not results:
    raise ValueError("FOLDS_TO_RUN è vuoto")
end = datetime.now(ZoneInfo("Europe/Rome"))
elapsed = end - start
hours, remainder = divmod(elapsed.seconds, 3600)
minutes, seconds = divmod(remainder, 60)
summary = {
    "fold_results": results,
    "mean_test_loss": sum(r["test_loss"] for r in results) / len(results),
    "mean_test_accuracy": sum(r["test_acc"] for r in results) / len(results),
    "best_fold_by_val_loss": min(results, key=lambda r: r["val_loss"])["fold"],
    "elapsed_hms": f"{hours:02d}:{minutes:02d}:{seconds:02d}",
}
(RUN_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(json.dumps(summary, indent=2))
print(f"File salvati in: {RUN_DIR}")

write_log(f"RUN FINISHED AT {end.strftime('%H:%M:%S')}")
write_log(f"ELAPSED {summary['elapsed_hms']}")
write_log(f"MEAN LOSS IN FOLDS: {summary['mean_test_loss']}")
write_log(f"MEAN ACCURACY IN FOLDS: {summary['mean_test_accuracy']}")
best_fold = min(results, key=lambda result: result["val_loss"])
write_log(
    f"BEST FOLD: {best_fold['fold']}\n"
    f"\tEPOCH: {best_fold['best_epoch']}\n"
    f"\tVAL LOSS: {best_fold['val_loss']}\n"
    f"\tVAL ACCURACY: {best_fold['val_acc']}\n"
    f"\tTEST LOSS: {best_fold['test_loss']}\n"
    f"\tTEST ACCURACY: {best_fold['test_acc']}\n"
)
print(f"Log locale: {LOG_PATH}")


In [ ]:
# Crea un unico archivio da scaricare ed estrarre nella root di GENSKETCH.
# La struttura interna sarà models/<run>/, logs/<run>/.
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

archive_path = OUTPUT_ROOT / f"{RUN_DIR.name}_per_progetto.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for pattern, target_dir in (
        ("*.pt", "models"),
        ("*.csv", "logs"),
        ("*.log", "logs"),
        ("*.json", "logs"),
    ):
        for file_path in sorted(RUN_DIR.glob(pattern)):
            destination = f"{target_dir}/{RUN_DIR.name}/{file_path.name}"
            archive.write(file_path, arcname=destination)

print(f"Archivio pronto: {archive_path}")
print("Scarica la ZIP e decomprimila nella root di GENSKETCH.")
print("Se il link non funziona, usa il pannello Output di Kaggle.")
display(FileLink(str(archive_path)))


## Uso su Kaggle

1. Crea un Dataset **privato** caricando una ZIP con struttura data/photo, data/sketch e data/index.json (Kaggle estrae le ZIP mantenendo le directory). Se carichi solo alcune categorie, includi comunque index.json originale per conservare gli ID globali. Verifica i termini d'uso di Sketchy prima di condividere pubblicamente i dati.
2. Crea un Notebook su Kaggle; dal menu File usa Import Notebook per caricare questo file .ipynb. In alternativa incolla le celle di codice in ordine.
3. Nel pannello Input usa Add Input e collega il tuo Dataset. In Session options imposta Accelerator su GPU. Avvia le celle dall'alto e controlla che la prima mostri device: cuda e il percorso dati corretto.
4. I risultati appaiono in /kaggle/working/runs/. Salva una versione del Notebook ed esporta checkpoint, CSV e JSON dall'Output. Se il tempo della sessione è insufficiente per cinque fold, modifica FOLDS_TO_RUN, esegui uno o più fold per sessione e conserva gli output delle diverse versioni.

Nota: per valutare nuove architetture scegli in base alla validazione; i risultati di test riportati per fold servono a stimare la prestazione finale. L'output a 125 classi è mantenuto se l'indice originale contiene la mappa completa.
